# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/darksider747/flyrank-1st/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions
Finding 1: ML Appendix - Feature Importance for Health Score

The paper reports that a Random Forest model predicting Health Score
found Average Position to be the #1 predictor at 43% importance,
followed by Impressions (32%) and Scroll Depth (15%).

Methodology question: Health Score is explicitly defined earlier in the
paper as "Impressions (30 pts) + position (30 pts) + CTR (20 pts) +
scroll depth (20 pts)." Since Position, Impressions, and Scroll Depth
are literally mathematical ingredients of the Health Score itself, is
it meaningful to call this "feature importance" - or is the model
essentially rediscovering its own ingredients? This resembles the
leakage pattern I tested myself in Week 3, where including a feature
that's directly derived from the label produced an artificially
perfect (and meaningless) result. The paper does partially acknowledge
this ("the target itself is partly constructed from some of these
inputs, so importance is descriptive rather than causal"), which is a
responsible disclosure - but a reader skimming just the bar chart might
still walk away thinking Position "predicts" Health in some external,
causal sense. A stronger version of this analysis might instead try to
predict something genuinely separate from Health Score's own formula -
like future traffic growth - using Position as an input feature, to see
if the relationship still holds outside of the metric's own definition.*

Finding 2: Myth #5 - "AI-Generated Content Is Penalized" (DEBUNKED)

The paper concludes: "This portfolio does not show a blanket penalty
tied only to AI use" after comparing content performance across
different AI writing models (OpenAI vs Gemini) within matched age
cohorts.

Methodology question: The original myth being tested is whether AI-
generated content is penalized compared to human-written content. But
the actual comparison performed only measures AI model A (OpenAI)
against AI model B (Gemini) - it never compares AI-written content
against human-written content at all. The paper even states the
dataset is "almost all AI-generated across 5 different models," which
means there may not be a meaningful human-written comparison group
available in this data at all. Does comparing two different AI systems
against each other actually test the original claim ("AI content is
penalized" - implicitly, versus human content)? A more direct test of
the original myth would need a genuine AI-vs-human comparison group,
not just an AI-vs-AI comparison. This doesn't mean the paper's
underlying observation is wrong (that model/process quality matters
more than which AI wrote it) - but the specific myth-debunking claim
may be broader than what this particular comparison can actually
support.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, subprocess

REPO_URL = "https://github.com/darksider747/flyrank-1st"
REPO_DIR = "/content/flyrank-1st"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("Now in:", os.getcwd())

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df)} rows")
print(df[["trend_direction", "trend_pct", "impressions_90d"]].head(10))

Now in: /content/flyrank-1st
Loaded 30000 rows
  trend_direction  trend_pct  impressions_90d
0            down      -41.4             3803
1            down      -57.7            15320
2            down      -60.9            12581
3          stable      -13.8            11751
4            down      -34.7            19140
5            down      -38.9             3970
6            down      -92.3               20
7          stable        0.6             1724
8            down      -58.8            32574
9            down      -29.2             1240


## 2. My model under an honest split (before/after)
Auditing my Week 5 setup for timing overlap:

My label (is_declining, from trend_direction) is based on trend_pct,
which - per the research paper's own definition of trend direction -
compares the most recent 30 days against the previous 30 days.

My Week 5 features included impressions_90d, a 90-day total. Since 90
days would typically include the most recent 30-day window used to
calculate trend_pct, there is a real risk that this feature partially
overlaps in time with the label itself - a subtler version of the
leakage issue I tested directly in Week 3.

I cannot fully confirm the exact date boundaries from this pre-aggregated
starter CSV alone (unlike the raw daily warehouse data from Week 3), but
this is a legitimate methodology risk worth addressing rather than
ignoring. As a precaution, I will retrain my model without impressions_90d
and compare the result - this is my "before vs after" honest-split
improvement for this week.

Before/After comparison (client_holdout split, same test set, Precision@50):

| Version                          | Precision@50 |
|-----------------------------------|--------------|
| Week 5 model (with impressions_90d) | 0.780      |
| Week 6 model (without impressions_90d) | 0.560   |
| Week 4 baseline rule              | 0.500        |

Removing impressions_90d caused a substantial drop in performance
(0.780 -> 0.560), which supports my suspicion that this feature likely
had timing overlap with the trend_direction label - a meaningful chunk
of my Week 5 result may have come from this overlap rather than genuine
predictive skill.

However, the honest, stricter version of the model (0.560) still beats
the Week 4 baseline (0.500), just by a smaller, more modest margin than
originally reported. This is a more trustworthy result: the model does
appear to add some real value beyond the hand-written rule, but the
improvement is more modest than my Week 5 notebook claimed. I am
revising my Week 5 claim accordingly in Section 4 of this notebook.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

tier_avg_ctr = df.groupby("position_tier")["ctr"].transform("mean")
df["ctr_below_tier"] = df["ctr"] < (0.7 * tier_avg_ctr)
df["is_stale"] = df["days_since_last_update"] >= 91
df["baseline_score"] = df["ctr_below_tier"].astype(int) + df["is_stale"].astype(int)

from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print(f"Train: {len(train_df)} rows, {train_df['client_id'].nunique()} clients")
print(f"Test: {len(test_df)} rows, {test_df['client_id'].nunique()} clients")
from sklearn.ensemble import RandomForestClassifier

def precision_at_k(df_scored, score_col, label_col, k=50):
    top_k = df_scored.sort_values(score_col, ascending=False).head(k)
    return top_k[label_col].mean()

before_features = ["ctr", "avg_position", "days_since_last_update", "impressions_90d", "word_count"]

train_X_before = train_df[before_features].fillna(train_df[before_features].median())
test_X_before = test_df[before_features].fillna(train_df[before_features].median())

model_before = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model_before.fit(train_X_before, train_df["is_declining"])

test_df["prob_before"] = model_before.predict_proba(test_X_before)[:, 1]
p50_before = precision_at_k(test_df, "prob_before", "is_declining", k=50)
print(f"BEFORE (with impressions_90d) Precision@50: {p50_before:.3f}")
after_features = ["ctr", "avg_position", "days_since_last_update", "word_count"]

train_X_after = train_df[after_features].fillna(train_df[after_features].median())
test_X_after = test_df[after_features].fillna(train_df[after_features].median())

model_after = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model_after.fit(train_X_after, train_df["is_declining"])

test_df["prob_after"] = model_after.predict_proba(test_X_after)[:, 1]
p50_after = precision_at_k(test_df, "prob_after", "is_declining", k=50)
print(f"AFTER (without impressions_90d) Precision@50: {p50_after:.3f}")

print(f"\nBaseline (Week 4 rule) Precision@50: {precision_at_k(test_df, 'baseline_score', 'is_declining', k=50):.3f}")

Train: 22885 rows, 24 clients
Test: 7115 rows, 8 clients
BEFORE (with impressions_90d) Precision@50: 0.780
AFTER (without impressions_90d) Precision@50: 0.560

Baseline (Week 4 rule) Precision@50: 0.500


## 3. Leakage audit

Leakage audit - checking each feature against the leakage checklist:

1. ctr - a rate calculated from the page's own click/impression history.
   Not derived from the label, not future data. SAFE.

2. avg_position - the page's average search ranking. Independent
   measurement, not derived from trend_direction. SAFE.

3. days_since_last_update - a simple time calculation (today minus last
   edit date). Not related to the label's calculation at all. SAFE.

4. word_count - a static content property, unrelated to time-based
   trend or traffic. SAFE.

5. impressions_90d - REMOVED after my Section 2 audit, due to likely
   timing overlap with the trend_direction label window (both plausibly
   drawing from the same recent 90-day period).

Product-flag check (repeating my Week 4 check, now for this feature set):
no health_score, priority_score, or action_type columns are used or even
present in this dataset.

Train/test overlap check: confirmed zero client overlap between train
and test (client_holdout split), verified with code in Week 5 and reused
here.

Conclusion: after removing impressions_90d, the remaining 4 features
pass the leakage checklist. This is a stricter, more defensible feature
set than my original Week 5 version.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Confirm no product-flag columns anywhere in the dataset
unsafe_columns = [c for c in df.columns if "health_score" in c.lower() or "priority_score" in c.lower() or "action_type" in c.lower()]
print("Unsafe product-flag columns present:", unsafe_columns if unsafe_columns else "None found")

# Confirm zero client overlap between train and test
overlap = set(train_df["client_id"]) & set(test_df["client_id"])
print(f"Clients in both train and test (should be 0): {len(overlap)}")

# Final feature set used
print("Final features (after leakage audit):", after_features)

Unsafe product-flag columns present: None found
Clients in both train and test (should be 0): 0
Final features (after leakage audit): ['ctr', 'avg_position', 'days_since_last_update', 'word_count']


## 4. Claim rewrite

Claim rewrite:

Original Week 5 claim: "The Random Forest model found roughly 1.56x as
many truly declining pages in its top 50 as the baseline rule did,
tested fairly on the same 8 held-out clients."

Issue: This claim was based on a feature set that likely included
impressions_90d, which probably has timing overlap with the label's
own calculation window. This week's audit (Section 2) showed that
removing this feature drops Precision@50 from 0.780 to 0.560 - meaning
a meaningful part of the original 1.56x improvement was likely inflated
by this overlap, not genuine predictive skill.

Revised claim: "Under a leakage-audited feature set (excluding a
feature with likely timing overlap with the label), the Random Forest
model observed a Precision@50 of 0.560 on held-out clients, compared
to 0.500 for the Week 4 baseline rule - a modest, directional
improvement of roughly 1.12x. This is a more defensible estimate of
the model's real advantage than my original Week 5 report, though the
improvement is smaller than initially claimed."

I am also softening the earlier claim about which features "mattered
most" (Week 5's feature importance ranking), since impressions_90d was
previously reported as the top feature (0.326 importance) but has now
been removed from the model entirely due to the leakage concern. This
ranking should not be treated as a stable finding until re-verified
on a leakage-free feature set in a future notebook.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
old_improvement_ratio = 0.780 / 0.500
new_improvement_ratio = 0.560 / 0.500

print(f"Original (Week 5) improvement ratio: {old_improvement_ratio:.2f}x")
print(f"Revised (leakage-audited) improvement ratio: {new_improvement_ratio:.2f}x")

Original (Week 5) improvement ratio: 1.56x
Revised (leakage-audited) improvement ratio: 1.12x


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.